# ANON4REID - L3 RAD Dataset Generation (Kaggle Dual T4)

Generate the Level 3 (RAD — Realistic Anonymization by Diffusion) dataset using:
- **Stable Diffusion v1.5** conditioned on **ControlNet OpenPose**
- **LCM-LoRA** for 8-step fast inference
- Dual-GPU multiprocessing across `cuda:0` and `cuda:1`

Processes all 3 Market-1501 splits: `query` + `bounding_box_test` + `bounding_box_train` (~36,036 images).

Each image is upscaled from 64×128 → 256×512, pose-extracted via OpenPose, diffusion-anonymized (new identity, same pose), then downscaled back to 64×128.

**Resumable:** Skips already-processed images, safe to re-run after Kaggle session interruptions.

> **Note:** `controlnet_aux` is installed with `--no-deps` to avoid the `mediapipe.solutions` incompatibility on Python 3.12+. The worker script patches `controlnet_aux/__init__.py` to skip the broken `MediapipeFaceDetector` import before loading `OpenposeDetector`.

In [ ]:
!pip install diffusers transformers accelerate opencv-python Pillow numpy peft xformers
!pip install controlnet_aux --no-deps

In [ ]:
# Write the self-contained worker script to /kaggle/working/rad_worker.py
# This avoids Jupyter multiprocessing limitations — all logic is in a standalone .py file.
# Includes a patch for controlnet_aux + mediapipe incompatibility on Python 3.12+.
# Parameters match src/rad_pipeline.py exactly.

worker_script = '''
import os
import math
import importlib.util
import torch
import torch.multiprocessing as mp
from PIL import Image
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, LCMScheduler

# ── Patch controlnet_aux to avoid mediapipe import error on Python 3.12+ ─────
# mediapipe removed its 'solutions' namespace in newer versions, causing
# controlnet_aux/__init__.py to crash when importing MediapipeFaceDetector.
# We only need OpenposeDetector, so we remove the problematic import line.
spec = importlib.util.find_spec('controlnet_aux')
if spec and spec.origin:
    with open(spec.origin, 'r') as f:
        init_src = f.read()
    if 'from .mediapipe_face import MediapipeFaceDetector' in init_src:
        init_src = init_src.replace(
            'from .mediapipe_face import MediapipeFaceDetector\n', ''
        )
        with open(spec.origin, 'w') as f:
            f.write(init_src)
        print("Patched controlnet_aux: removed mediapipe_face import.")
    else:
        print("controlnet_aux __init__.py already patched or import line absent.")
else:
    raise ImportError("controlnet_aux package not found. Check pip install.")

from controlnet_aux import OpenposeDetector

# ── Constants ──────────────────────────────────────────────────────────────────
ORIGINAL_SIZE = (64, 128)  # (width, height) — PIL convention, matches 128x64 Market-1501
MODEL_SIZE    = (256, 512) # Upscaled to model input dimensions (width=256, height=512)

# ── Paths ──────────────────────────────────────────────────────────────────────
INPUT_DIR  = "/kaggle/input/datasets/rayiooo/reid_market-1501/Market-1501-v15.09.15"  # Change if Kaggle mounts differently
OUTPUT_DIR = "/kaggle/working/Market-1501-RAD"
SPLITS     = ["query", "bounding_box_test", "bounding_box_train"]

# ── RAD pipeline parameters (must match src/rad_pipeline.py exactly) ──────────
PROMPT                     = "a person, different appearance, full body, high quality, realistic"
NEGATIVE_PROMPT            = "blurry, low quality, distorted, noise, same person"
NUM_INFERENCE_STEPS        = 8
GUIDANCE_SCALE             = 1.5
CONTROLNET_CONDITIONING_SCALE = 0.8
HEIGHT                     = 512
WIDTH                      = 256


def preprocess(image: Image.Image) -> Image.Image:
    image = image.convert("RGB")
    return image.resize(MODEL_SIZE, Image.LANCZOS)


def postprocess(image: Image.Image) -> Image.Image:
    return image.resize(ORIGINAL_SIZE, Image.LANCZOS)


def extract_pose(image: Image.Image, detector: OpenposeDetector) -> Image.Image:
    return detector(image)


def load_rad_pipeline(device_id: int):
    # Load the full RAD pipeline (ControlNet + SD v1.5 + LCM-LoRA) on specified GPU.
    device = f"cuda:{device_id}"

    print(f"[{device}] Loading ControlNet OpenPose model...")
    controlnet = ControlNetModel.from_pretrained(
        "lllyasviel/control_v11p_sd15_openpose",
        torch_dtype=torch.float16,
    )

    print(f"[{device}] Loading Stable Diffusion v1.5 with ControlNet...")
    pipe = StableDiffusionControlNetPipeline.from_pretrained(
        "stable-diffusion-v1-5/stable-diffusion-v1-5",
        controlnet=controlnet,
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to(device)

    print(f"[{device}] Loading LCM-LoRA for fast inference...")
    pipe.load_lora_weights("latent-consistency/lcm-lora-sdv1-5")
    pipe.scheduler = LCMScheduler.from_config(pipe.scheduler.config)

    # Memory optimization: try xformers, fallback to attention slicing
    try:
        pipe.enable_xformers_memory_efficient_attention()
        print(f"[{device}] xformers memory-efficient attention enabled.")
    except Exception:
        pipe.enable_attention_slicing()
        print(f"[{device}] xformers unavailable — using attention slicing fallback.")

    pipe.set_progress_bar_config(disable=True)

    print(f"[{device}] Loading OpenPose detector...")
    detector = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")

    print(f"[{device}] RAD pipeline ready.")
    return pipe, detector


def process_gpu(gpu_id: int, task_list: list):
    # Process assigned images on a single GPU. Skip existing outputs for resumability.
    device_str = f"cuda:{gpu_id}"
    print(f"[{device_str}] Worker started — {len(task_list)} images assigned.")

    pipe, detector = load_rad_pipeline(gpu_id)

    # Skip already-processed images (resumability)
    pending = [(in_path, out_path) for in_path, out_path in task_list
               if not os.path.exists(out_path)]
    skipped = len(task_list) - len(pending)
    if skipped:
        print(f"[{device_str}] Skipped {skipped} already-processed images.")
    print(f"[{device_str}] Processing {len(pending)} images...")

    processed = 0
    errors = 0
    milestone = max(1, len(pending) // 10)

    for in_path, out_path in pending:
        try:
            img = Image.open(in_path).convert("RGB")
            img_up = preprocess(img)
            pose = extract_pose(img_up, detector)

            result = pipe(
                prompt=PROMPT,
                negative_prompt=NEGATIVE_PROMPT,
                image=pose,
                height=HEIGHT,
                width=WIDTH,
                num_inference_steps=NUM_INFERENCE_STEPS,
                guidance_scale=GUIDANCE_SCALE,
                controlnet_conditioning_scale=CONTROLNET_CONDITIONING_SCALE,
            ).images[0]

            result_down = postprocess(result)
            result_down.save(out_path)
            processed += 1

        except Exception as e:
            print(f"[{device_str}] Error on {in_path}: {e}")
            errors += 1

        if processed > 0 and processed % milestone == 0:
            pct = processed / len(pending) * 100
            print(f"[{device_str}] Progress: {processed}/{len(pending)} ({pct:.0f}%)")

    print(f"[{device_str}] Finished. Processed: {processed}, Errors: {errors}")


if __name__ == "__main__":
    try:
        mp.set_start_method("spawn", force=True)
    except RuntimeError:
        pass

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    for split in SPLITS:
        os.makedirs(os.path.join(OUTPUT_DIR, split), exist_ok=True)

    # Collect all (input_path, output_path) tasks across all splits
    all_tasks = []
    for split in SPLITS:
        split_dir_in  = os.path.join(INPUT_DIR, split)
        split_dir_out = os.path.join(OUTPUT_DIR, split)
        if os.path.exists(split_dir_in):
            for fname in sorted(os.listdir(split_dir_in)):
                if fname.lower().endswith(".jpg"):
                    all_tasks.append(
                        (os.path.join(split_dir_in, fname),
                         os.path.join(split_dir_out, fname))
                    )
        else:
            print(f"Warning: split directory not found: {split_dir_in}")

    total = len(all_tasks)
    print(f"Total images to process: {total}")

    if total == 0:
        print(f"No images found at {INPUT_DIR}. Check dataset mount path.")
    else:
        # Split tasks at midpoint between cuda:0 and cuda:1
        midpoint = math.ceil(total / 2)
        tasks_gpu0 = all_tasks[:midpoint]
        tasks_gpu1 = all_tasks[midpoint:]

        print(f"GPU 0 tasks: {len(tasks_gpu0)}, GPU 1 tasks: {len(tasks_gpu1)}")

        p0 = mp.Process(target=process_gpu, args=(0, tasks_gpu0))
        p1 = mp.Process(target=process_gpu, args=(1, tasks_gpu1))

        p0.start()
        p1.start()
        p0.join()
        p1.join()

        print("All GPU workers finished!")
'''

with open('/kaggle/working/rad_worker.py', 'w') as f:
    f.write(worker_script)

print('Wrote /kaggle/working/rad_worker.py')


In [ ]:
!python /kaggle/working/rad_worker.py

In [ ]:
import os

OUTPUT_DIR = "/kaggle/working/Market-1501-RAD"
SPLITS = ["query", "bounding_box_test", "bounding_box_train"]

print("=== Verification: per-split file countrads ===")
total = 0
for split in SPLITS:
    split_dir = os.path.join(OUTPUT_DIR, split)
    if os.path.exists(split_dir):
        count = len([f for f in os.listdir(split_dir) if f.lower().endswith(".jpg")])
        print(f"  {split}: {count} images")
        total += count
    else:
        print(f"  {split}: directory not found!")

print(f"\nTotal RAD images: {total}  (expected ~36,036)")

In [ ]:
import shutil

OUTPUT_DIR = "/kaggle/working/Market-1501-RAD"

print("Zipping Market-1501-RAD...")
archive_path = shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
print(f"Done! Archive: {archive_path}")
print(f"Download {OUTPUT_DIR}.zip from the Kaggle output panel.")
print("Then extract into data/processed/Market-1501-RAD/ on your local machine.")